In [83]:
import numpy as np
import math
import matplotlib.pyplot as plt
%matplotlib inline

In [1]:
a = 2
b = 4
c = a*b
print(c)

h = 0.0001
a = a + h
b = 4
c1 = a*b
print(c1)

derivate = (c1-c)/h
print(derivate) #print 4

8
8.0004
4.000000000008441


In [ ]:

class Value:
    
    def __init__(self, data, label="", op="", children=()):
        self.data = data
        self.label = label
        self._op = op
        self._prev = set(children)

        self.grad = 0.0
        self._backward = lambda: None
        
    def __add__(self, data, label=""):
        if isinstance(data, (int, float)):
            data = Value(data)
            
        out = Value(self.data + data.data, label, op="+", children=(self, data))
        
        def backward():
            self.grad += 1.0 * out.grad
            data.grad += 1.0 * out.grad        
        out._backward = backward
        
        return out
    
    def __radd__(self, other):
        return self.__add__(other)
    
    def __sub__(self, other):
        if isinstance(other, (int, float)):
            other = Value(other)
        return self.__add__(other * -1)
    
    def __rsub__(self, other):
        return self.__sub__(other)
    
    def __mul__(self, data, label=""):
        if isinstance(data, (int, float)):
            data = Value(data)
            
        out = Value(self.data * data.data, label, op="*", children=(self, data))
        
        def backward():
            self.grad += out.grad * data.data
            data.grad += out.grad * self.data
        out._backward = backward
        
        return out
    
    def __rmul__(self, other):
        return self.__mul__(other)
    
    def __pow__(self, other):
        if not isinstance(other, int):
            raise Exception("power should be of an integer")
        
        out = Value(data=self.data ** other, op="**", children=(self,))
        def backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = backward
        
        return out
    
    def __truediv__(self, other):
        # a/b = a*(b**-1)
        return self.__mul__(other**-1)
    
    
    def tanh(self, label=""):
        x = self.data
        tanh = (math.exp(2*x)-1)/(math.exp(2*x)+1)
        out = Value(tanh, op='tanh', children=(self,))
        
        def backward():
            self.grad += (1 - tanh**2) * out.grad
        
        out._backward = backward
        
        return out
    
    def backward(self):
        self.grad = 1.0
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        for n in reversed(topo):
            n._backward()
    
    def __repr__(self):
        string = f"Value(data={self.data}"
        if self.label:
            string += f", label=\"{self.label}\""
        string += ")"
        return string
    


In [561]:
b = Value(4, label='b')
a = Value(2, label='a')
print(a, b)
mul = a * b; mul.label = "a*b"
add = a + b
add.label = "a+b"
print(mul, add)
a = Value(11); a.label = 'a'
b = a**2
print(a, b)
b.backward()
print(a.grad)
print(a,2,a/2)
print(a,b,a/b)
print(a-1, 1-a)

Value(data=2, label="a") Value(data=4, label="b")
Value(data=8, label="a*b") Value(data=6, label="a+b")
Value(data=11, label="a") Value(data=121)
22.0
Value(data=11, label="a") 2 Value(data=5.5)
Value(data=11, label="a") Value(data=121) Value(data=0.09090909090909091)
Value(data=10) Value(data=10)


In [562]:
class Neuron:
    
    def __init__(self, nin): #nin is the no of input weights
        self.w = [Value(np.random.uniform(-1,1)) for i in range(nin)]
        self.b = Value(np.random.uniform(-1,1))
    
    def __call__(self, x):
        # w.x + b
        return (sum([wi*xi for wi,xi in zip(self.w, x)], self.b)).tanh()
    
    def parameters(self):
        return self.w + [self.b]
    
x = [2.0, 3.0]
n = Neuron(2)
print(n(x))
print(n.parameters())

Value(data=0.9942883851778946)
[Value(data=0.8895904139731405), Value(data=0.2926095999322964), Value(data=0.2707607606938176)]


In [563]:
class Layer:
    # input to this layer and output from this layer
    def __init__(self, nin, nout):
        self.nin = nin
        self.nout = nout
        self.neurons = [Neuron(nin) for i in range(nout)]
        
    def __call__(self, x):
        return [neuron(x) for neuron in self.neurons]
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
    
class MLP:
    def __init__(self, nodes_len): # first value of nodes_len is the input's dim len, others are all node sizes
        self.layers = [Layer(nodes_len[i-1], nodes_len[i])  for i in range(1, len(nodes_len))]
        
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x
    
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
    
x = [2.0, 3.0, -1.0]
mlp = MLP([len(x), 4, 4, 1])
print(mlp(x))
print(mlp.parameters())
print(len(mlp.parameters()))

Value(data=0.6408937683737775)
[Value(data=-0.3837701683400381), Value(data=0.6909284998299194), Value(data=-0.014589133009233146), Value(data=0.49161277098502154), Value(data=0.07741304920399594), Value(data=0.8461218622421016), Value(data=0.8600562643612848), Value(data=-0.2473688466357229), Value(data=0.7064697234170065), Value(data=0.4128868666530803), Value(data=-0.7759499201315829), Value(data=-0.5916808811882175), Value(data=-0.31454581817970606), Value(data=0.5125768950611325), Value(data=0.20452928317061025), Value(data=-0.06764271497569418), Value(data=-0.9550566776392397), Value(data=-0.7488832290235885), Value(data=0.6982402548549913), Value(data=-0.2973327216809194), Value(data=-0.5088524069731744), Value(data=0.8031263116566134), Value(data=0.1417386430671601), Value(data=0.5787704410410586), Value(data=0.9596369101310009), Value(data=0.7645770523807234), Value(data=0.9516856050309979), Value(data=-0.6412807921722548), Value(data=0.7665523660529541), Value(data=0.16543175

In [569]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]
ys = [1.0, -1.0, -1.0, 1.0]
mlp = MLP([len(x), 4, 4, 1])

y_pred = [mlp(x) for x in xs]
print(y_pred)

loss = sum([(yp - y)**2 for yp, y in zip(y_pred, ys)])
print(loss)

loss.backward()

print([w.grad for w in mlp.layers[0].neurons[0].w])

[Value(data=-0.7216632402350681), Value(data=0.010958421463128504), Value(data=-0.04565283037651871), Value(data=-0.6357987608106859)]
Value(data=7.5727773487420595)
[2.5379591210251986, 0.2989241068618832, 1.7305546429328165]


In [586]:
mlp = MLP([3, 4, 4, 1])

In [714]:
epochs = 20
for i in range(epochs):
    
    # forward pass
    y_pred = [mlp(x) for x in xs]
    
    loss = sum([(yp - y)**2 for yp, y in zip(y_pred, ys)])
    mlp.zero_grad()
    loss.backward()
    
    # gradient update
    for p in mlp.parameters():
        p.data += -0.01*p.grad
        
    print(i, loss.data)
    
    

0 0.002763968556456661
1 0.0027627679309590145
2 0.0027615683290051858
3 0.0027603697492957647
4 0.002759172190533582
5 0.002757975651423678
6 0.0027567801306732447
7 0.0027555856269916054
8 0.002754392139090347
9 0.0027531996656831436
10 0.0027520082054858785
11 0.0027508177572165603
12 0.0027496283195953966
13 0.0027484398913446965
14 0.0027472524711889294
15 0.002746066057854714
16 0.0027448806500707914
17 0.002743696246568035
18 0.0027425128460794616
19 0.0027413304473401933


In [ ]:
y_pred = [mlp(x) for x in xs]
print(y_pred)

[Value(data=0.9738626261234572), Value(data=-0.9779139285406123), Value(data=-0.9708369706322761), Value(data=0.9731912346141534)]
